# 02 — Preprocessing
Explore and validate the preprocessing pipeline before training.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import DataPreprocessor

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
DATA_PATH = '../data/raw/dataset.csv'
TARGET_COL = 'target'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Summary before preprocessing
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Numerical columns: {df.select_dtypes(include="number").shape[1]}')
print(f'Categorical columns: {df.select_dtypes(include=["object","category"]).shape[1]}')

In [ ]:
# Fit and transform
preprocessor = DataPreprocessor()
X_transformed, y = preprocessor.fit_transform(df, TARGET_COL)

print(f'Task: {preprocessor.task_type}')
print(f'X shape after transform: {X_transformed.shape}')
print(f'Numerical columns: {preprocessor.numerical_cols}')
print(f'Categorical columns: {preprocessor.categorical_cols}')

if preprocessor.label_encoder:
    print(f'Encoded classes: {preprocessor.label_encoder.classes_}')

In [ ]:
# Feature names after transformation
X_df = pd.DataFrame(X_transformed, columns=preprocessor.feature_names_out)
print(f'Features ({len(preprocessor.feature_names_out)}):')
print(preprocessor.feature_names_out)
X_df.head()

In [ ]:
# Distribution of transformed numerical features
num_features = [f for f in preprocessor.feature_names_out if not any(c in f for c in preprocessor.categorical_cols)]
if num_features:
    n_cols = min(3, len(num_features))
    n_rows = (len(num_features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(num_features):
        axes[i].hist(X_df[col], bins=30, color='steelblue', alpha=0.8)
        axes[i].set_title(f'{col} (standardized)')
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Feature distributions after StandardScaler', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Transform new data (without target)
new_data = df.drop(columns=[TARGET_COL]).sample(5, random_state=42).reset_index(drop=True)
X_new = preprocessor.transform(new_data)
print(f'New data shape: {X_new.shape}')
pd.DataFrame(X_new, columns=preprocessor.feature_names_out).head()

In [ ]:
# Save preprocessor
preprocessor.save('../models/preprocessor.pkl')
print('Preprocessor saved to models/preprocessor.pkl')